# Hardcore Information-Theory Analysis of RBPMS + FISH Flatmount Retina Spatial Maps

**Goal:** quantify how much spatial/topographic information each mRNA probe carries across flatmount mouse retinas.

This notebook assumes you have already converted images into a **per-cell table**, where each row is one RBPMS+ retinal ganglion cell with retinal position and FISH signal for one probe.

Main outputs:

1. Entropy of each probe's spatial expression map
2. Mutual information between probe expression and retinal location
3. Normalized mutual information
4. Null-model shuffled z-scores and empirical p-values
5. Jensen-Shannon divergence between probe spatial maps
6. ML clustering/embedding of probes by spatial information features
7. Ranked table of most spatially informative probes


## Library guide

- `pandas`: cell-level tables and result tables  
- `numpy`: numerical arrays, binning, randomization, probability distributions  
- `scipy`: entropy and Jensen-Shannon distance  
- `sklearn`: mutual information, scaling, PCA, clustering, silhouette score  
- `matplotlib`: plots  
- `seaborn`: heatmaps and clustered visualizations  
- `pathlib`: clean file paths  
- Optional `umap-learn`: nonlinear 2D embedding of probes; PCA is used if UMAP is unavailable.


In [ ]:
# Core imports
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from scipy.stats import entropy
from scipy.spatial.distance import jensenshannon

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mutual_info_score, normalized_mutual_info_score
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import matplotlib.pyplot as plt
import seaborn as sns

try:
    import umap
    HAS_UMAP = True
except Exception:
    HAS_UMAP = False

sns.set_context("notebook")


## 1. Set paths and expected input format

Expected input CSV: one row per detected RBPMS+ RGC.

Required columns:

```text
probe_name       name of FISH probe / gene
retina_id        biological replicate / image ID
x                cell centroid x-coordinate
y                cell centroid y-coordinate
fish_intensity   FISH intensity measured in/around each RGC
```

Optional columns:

```text
group                 control / injury / manipulation
rbpms_intensity        RBPMS intensity per cell
distance_from_onh      normalized distance from optic nerve head
angle                  polar angle around retina
region                 dorsal/ventral/nasal/temporal/central/peripheral etc.
fish_positive          binary 0/1 expression call
```

If `fish_positive` is missing, this notebook generates it using a per-probe percentile threshold.


In [ ]:
# ---------------- USER SETTINGS ----------------
DATA_PATH = Path("data/rbpms_fish_per_cell_table.csv")
OUTPUT_DIR = Path("outputs_information_theory")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

N_X_BINS = 8
N_Y_BINS = 8
N_INTENSITY_BINS = 4
FISH_POSITIVE_PERCENTILE = 80
N_SHUFFLES = 500
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

print("Output folder:", OUTPUT_DIR.resolve())


## 2. Load data or generate a simulated example

This cell loads your real per-cell table. If the file does not exist, it creates simulated data with 50 probes and 3 retinas per probe so the notebook runs end-to-end.


In [ ]:
def simulate_retina_fish_dataset(n_probes=50, n_retinas_per_probe=3, cells_per_retina=1500, seed=42):
    rng = np.random.default_rng(seed)
    rows = []
    motifs = ["uniform", "central", "peripheral", "dorsal", "ventral", "nasal", "temporal", "patchy"]
    probe_names = [f"Probe_{i:02d}" for i in range(1, n_probes + 1)]

    for p_idx, probe in enumerate(probe_names):
        motif = motifs[p_idx % len(motifs)]
        for r in range(n_retinas_per_probe):
            retina_id = f"{probe}_retina_{r+1}"
            theta = rng.uniform(0, 2*np.pi, cells_per_retina)
            radius = np.sqrt(rng.uniform(0, 1, cells_per_retina))
            x = radius * np.cos(theta)
            y = radius * np.sin(theta)
            dist = radius
            angle = theta

            if motif == "uniform":
                field = np.ones(cells_per_retina) * 0.5
            elif motif == "central":
                field = np.exp(-dist**2 / 0.20)
            elif motif == "peripheral":
                field = dist**2
            elif motif == "dorsal":
                field = (y + 1) / 2
            elif motif == "ventral":
                field = (1 - y) / 2
            elif motif == "nasal":
                field = (x + 1) / 2
            elif motif == "temporal":
                field = (1 - x) / 2
            else:
                centers = np.array([[0.4, 0.4], [-0.45, 0.2], [0.1, -0.55]])
                field = np.zeros(cells_per_retina)
                for cx, cy in centers:
                    field += np.exp(-((x-cx)**2 + (y-cy)**2) / 0.05)
                field = field / field.max()

            fish = 100 + 500*field + rng.normal(0, 80, cells_per_retina)
            fish = np.clip(fish, 0, None)
            rbpms = rng.normal(1000, 120, cells_per_retina)

            for i in range(cells_per_retina):
                rows.append({
                    "probe_name": probe,
                    "retina_id": retina_id,
                    "motif_simulated": motif,
                    "x": x[i],
                    "y": y[i],
                    "distance_from_onh": dist[i],
                    "angle": angle[i],
                    "rbpms_intensity": rbpms[i],
                    "fish_intensity": fish[i],
                    "group": "control"
                })
    return pd.DataFrame(rows)

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    print("Loaded real data:", DATA_PATH)
else:
    df = simulate_retina_fish_dataset()
    print("DATA_PATH not found. Using simulated example data.")

print(df.shape)
display(df.head())


## 3. Validate and standardize the cell table

This step checks required columns, removes incomplete rows, creates spatial bins, normalizes FISH intensity within retina, and creates a binary FISH-positive call if needed.


In [ ]:
required = ["probe_name", "retina_id", "x", "y", "fish_intensity"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

analysis_df = df.dropna(subset=required).copy()

analysis_df["fish_z_within_retina"] = analysis_df.groupby("retina_id")["fish_intensity"].transform(
    lambda s: (s - s.mean()) / (s.std() + 1e-9)
)

if "fish_positive" not in analysis_df.columns:
    thresholds = analysis_df.groupby("probe_name")["fish_intensity"].transform(
        lambda s: np.percentile(s, FISH_POSITIVE_PERCENTILE)
    )
    analysis_df["fish_positive"] = (analysis_df["fish_intensity"] >= thresholds).astype(int)

analysis_df["x_bin"] = pd.cut(analysis_df["x"], bins=N_X_BINS, labels=False, include_lowest=True)
analysis_df["y_bin"] = pd.cut(analysis_df["y"], bins=N_Y_BINS, labels=False, include_lowest=True)
analysis_df["spatial_bin"] = analysis_df["x_bin"].astype(str) + "_" + analysis_df["y_bin"].astype(str)

analysis_df["fish_intensity_bin"] = analysis_df.groupby("probe_name")["fish_z_within_retina"].transform(
    lambda s: pd.qcut(s.rank(method="first"), q=N_INTENSITY_BINS, labels=False, duplicates="drop")
).astype(int)

print("Analysis rows:", len(analysis_df))
print("Probes:", analysis_df["probe_name"].nunique())
print("Retinas:", analysis_df["retina_id"].nunique())
display(analysis_df.head())


## 4. Helper functions for information-theory metrics

- **Spatial entropy:** how spread out positive/expression-weighted signal is across retina bins.  
- **Mutual information:** how much knowing expression reduces uncertainty about retinal location.  
- **Normalized mutual information:** MI scaled to make probes more comparable.  
- **Null z-score:** how much larger observed MI is than shuffled-position MI.  
- **Jensen-Shannon divergence:** symmetric distance between probe spatial probability maps.


In [ ]:
def probability_vector(counts, all_bins):
    v = counts.reindex(all_bins, fill_value=0).astype(float).values
    total = v.sum()
    if total <= 0:
        return np.ones(len(all_bins)) / len(all_bins)
    return v / total


def safe_entropy(p):
    p = np.asarray(p, dtype=float)
    p = p[p > 0]
    return entropy(p, base=2)


def normalized_entropy(p):
    h = safe_entropy(p)
    hmax = np.log2(len(p)) if len(p) > 1 else 1
    return h / hmax


def mutual_information_discrete(a, b):
    return mutual_info_score(a, b)


def nmi_discrete(a, b):
    return normalized_mutual_info_score(a, b)


def probe_spatial_probability_table(data, value_col="fish_positive"):
    all_bins = sorted(data["spatial_bin"].dropna().unique())
    maps = []
    for probe, sub in data.groupby("probe_name"):
        if value_col == "fish_positive":
            counts = sub.loc[sub[value_col] == 1, "spatial_bin"].value_counts()
        else:
            counts = sub.groupby("spatial_bin")[value_col].sum()
        p = probability_vector(counts, all_bins)
        maps.append(pd.Series(p, index=all_bins, name=probe))
    return pd.DataFrame(maps)


## 5. Compute per-probe information metrics

For each probe, this computes how spatially concentrated the FISH-positive cells are, how much expression predicts spatial bin, and whether this exceeds shuffled null models.


In [ ]:
def compute_probe_metrics(data, n_shuffles=500, rng=None):
    if rng is None:
        rng = np.random.default_rng(0)

    all_bins = sorted(data["spatial_bin"].dropna().unique())
    results = []

    for probe, sub in data.groupby("probe_name"):
        sub = sub.dropna(subset=["spatial_bin", "fish_intensity_bin", "fish_positive"]).copy()
        if len(sub) < 50:
            continue

        pos_counts = sub.loc[sub["fish_positive"] == 1, "spatial_bin"].value_counts()
        p_pos_space = probability_vector(pos_counts, all_bins)

        mi_intensity_space = mutual_information_discrete(sub["fish_intensity_bin"], sub["spatial_bin"])
        nmi_intensity_space = nmi_discrete(sub["fish_intensity_bin"], sub["spatial_bin"])
        mi_binary_space = mutual_information_discrete(sub["fish_positive"], sub["spatial_bin"])
        nmi_binary_space = nmi_discrete(sub["fish_positive"], sub["spatial_bin"])

        null_mi = []
        spatial_values = sub["spatial_bin"].values.copy()
        expr_values = sub["fish_intensity_bin"].values.copy()
        for _ in range(n_shuffles):
            null_mi.append(mutual_information_discrete(expr_values, rng.permutation(spatial_values)))
        null_mi = np.asarray(null_mi)

        z = (mi_intensity_space - null_mi.mean()) / (null_mi.std() + 1e-12)
        p_emp = (np.sum(null_mi >= mi_intensity_space) + 1) / (n_shuffles + 1)

        results.append({
            "probe_name": probe,
            "n_cells": len(sub),
            "n_retinas": sub["retina_id"].nunique(),
            "positive_fraction": sub["fish_positive"].mean(),
            "positive_spatial_entropy_bits": safe_entropy(p_pos_space),
            "positive_spatial_entropy_norm": normalized_entropy(p_pos_space),
            "mi_intensity_vs_space": mi_intensity_space,
            "nmi_intensity_vs_space": nmi_intensity_space,
            "mi_binary_vs_space": mi_binary_space,
            "nmi_binary_vs_space": nmi_binary_space,
            "null_mi_mean": null_mi.mean(),
            "null_mi_sd": null_mi.std(),
            "mi_shuffle_z": z,
            "mi_empirical_p": p_emp,
        })

    return pd.DataFrame(results).sort_values("mi_shuffle_z", ascending=False)

probe_metrics = compute_probe_metrics(analysis_df, n_shuffles=N_SHUFFLES, rng=rng)
probe_metrics.to_csv(OUTPUT_DIR / "per_probe_information_metrics.csv", index=False)
display(probe_metrics.head(15))


## 6. Visualize the strongest spatially informative probes

High `mi_shuffle_z` means expression is more spatially organized than expected after random shuffling.


In [ ]:
plt.figure(figsize=(9, 6))
top = probe_metrics.head(20).sort_values("mi_shuffle_z")
plt.barh(top["probe_name"], top["mi_shuffle_z"])
plt.xlabel("MI shuffle z-score")
plt.ylabel("Probe")
plt.title("Top spatially informative mRNA probes")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "top_probe_mi_zscores.png", dpi=300)
plt.show()


## 7. Build spatial probability maps for each probe

Each row is a probe. Each column is a retinal spatial bin. Values are probability mass from FISH-positive cells.


In [ ]:
probe_maps = probe_spatial_probability_table(analysis_df, value_col="fish_positive")
probe_maps.to_csv(OUTPUT_DIR / "probe_spatial_probability_maps.csv")

plt.figure(figsize=(12, 8))
sns.heatmap(probe_maps, cmap="viridis")
plt.xlabel("Retinal spatial bin")
plt.ylabel("Probe")
plt.title("Probe spatial probability maps")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "probe_spatial_probability_heatmap.png", dpi=300)
plt.show()


## 8. Jensen-Shannon divergence between probes

Jensen-Shannon divergence compares full spatial probability maps. Small values mean similar topographies; large values mean different topographies.


In [ ]:
probes = probe_maps.index.tolist()
jsd = pd.DataFrame(index=probes, columns=probes, dtype=float)

for p1 in probes:
    for p2 in probes:
        jsd.loc[p1, p2] = jensenshannon(probe_maps.loc[p1], probe_maps.loc[p2], base=2) ** 2

jsd.to_csv(OUTPUT_DIR / "probe_jensen_shannon_divergence_matrix.csv")

sns.clustermap(jsd, cmap="mako", figsize=(10, 10))
plt.savefig(OUTPUT_DIR / "probe_jsd_clustermap.png", dpi=300, bbox_inches="tight")
plt.show()


## 9. ML clustering of probes by information-theory features

This groups probes into spatial expression classes using information-theory summary features.


In [ ]:
feature_cols = [
    "positive_fraction",
    "positive_spatial_entropy_norm",
    "mi_intensity_vs_space",
    "nmi_intensity_vs_space",
    "mi_binary_vs_space",
    "nmi_binary_vs_space",
    "mi_shuffle_z",
]

features = probe_metrics.set_index("probe_name")[feature_cols].replace([np.inf, -np.inf], np.nan).dropna()
X = StandardScaler().fit_transform(features)

candidate_k = range(2, min(10, len(features)))
scores = {}
for k in candidate_k:
    labels = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init="auto").fit_predict(X)
    scores[k] = silhouette_score(X, labels)

best_k = max(scores, key=scores.get)
cluster_labels = KMeans(n_clusters=best_k, random_state=RANDOM_SEED, n_init="auto").fit_predict(X)

clustered_features = features.copy()
clustered_features["cluster"] = cluster_labels
clustered_features = clustered_features.sort_values(["cluster", "mi_shuffle_z"], ascending=[True, False])
clustered_features.to_csv(OUTPUT_DIR / "probe_information_theory_clusters.csv")

print("Best k by silhouette:", best_k)
display(clustered_features.head(20))


## 10. 2D embedding of probes

This plots probes in 2D based on their information-theory features. UMAP is used if installed; PCA is used as a fallback.


In [ ]:
if HAS_UMAP:
    reducer = umap.UMAP(random_state=RANDOM_SEED)
    emb = reducer.fit_transform(X)
    method = "UMAP"
else:
    reducer = PCA(n_components=2, random_state=RANDOM_SEED)
    emb = reducer.fit_transform(X)
    method = "PCA"

embed_df = pd.DataFrame(emb, columns=["dim1", "dim2"], index=features.index)
embed_df["cluster"] = cluster_labels
embed_df["mi_shuffle_z"] = features["mi_shuffle_z"]
embed_df.to_csv(OUTPUT_DIR / "probe_2d_embedding.csv")

plt.figure(figsize=(8, 6))
plt.scatter(embed_df["dim1"], embed_df["dim2"], c=embed_df["cluster"], s=70)
for probe, row in embed_df.iterrows():
    plt.text(row["dim1"], row["dim2"], probe, fontsize=7, alpha=0.8)
plt.xlabel(f"{method} 1")
plt.ylabel(f"{method} 2")
plt.title(f"Probe embedding from information-theory features ({method})")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "probe_information_embedding.png", dpi=300)
plt.show()


## 11. Optional density-stratified MI control

This asks whether probe expression carries spatial information beyond baseline RGC sampling density. It is a practical approximation to conditional mutual information.


In [ ]:
bin_counts = analysis_df.groupby(["retina_id", "spatial_bin"]).size().rename("rgc_density_bin_count").reset_index()
cond_df = analysis_df.merge(bin_counts, on=["retina_id", "spatial_bin"], how="left")
cond_df["density_quantile"] = pd.qcut(cond_df["rgc_density_bin_count"].rank(method="first"), q=4, labels=False)

cond_results = []
for probe, sub in cond_df.groupby("probe_name"):
    weighted_mis = []
    weights = []
    for _, ss in sub.groupby("density_quantile"):
        if ss["spatial_bin"].nunique() > 1 and ss["fish_intensity_bin"].nunique() > 1:
            weighted_mis.append(mutual_information_discrete(ss["fish_intensity_bin"], ss["spatial_bin"]))
            weights.append(len(ss))
    cmi_approx = np.average(weighted_mis, weights=weights) if weighted_mis else np.nan
    cond_results.append({"probe_name": probe, "density_stratified_mi_approx": cmi_approx})

cond_metrics = pd.DataFrame(cond_results)
probe_metrics_cond = probe_metrics.merge(cond_metrics, on="probe_name", how="left")
probe_metrics_cond.to_csv(OUTPUT_DIR / "per_probe_metrics_with_density_control.csv", index=False)
display(probe_metrics_cond.sort_values("density_stratified_mi_approx", ascending=False).head(15))


## 12. Final ranked output table

Recommended interpretation:

- High `mi_shuffle_z`: strong spatial organization beyond shuffled null
- Low `positive_spatial_entropy_norm`: localized/restricted expression
- High JSD to other probes: unique spatial pattern
- High density-stratified MI: spatial structure not explained by RGC density alone


In [ ]:
mean_jsd = jsd.replace(0, np.nan).mean(axis=1).rename("mean_jsd_to_other_probes")
final_table = probe_metrics_cond.set_index("probe_name").join(mean_jsd).reset_index()

rank_cols = [
    "probe_name",
    "n_cells",
    "n_retinas",
    "positive_fraction",
    "positive_spatial_entropy_norm",
    "mi_intensity_vs_space",
    "nmi_intensity_vs_space",
    "mi_shuffle_z",
    "mi_empirical_p",
    "density_stratified_mi_approx",
    "mean_jsd_to_other_probes",
]

final_table = final_table[rank_cols].sort_values("mi_shuffle_z", ascending=False)
final_table.to_csv(OUTPUT_DIR / "FINAL_ranked_probe_spatial_information_table.csv", index=False)
display(final_table.head(25))


## 13. Minimal conclusions template

Use this after running real data:

1. Probes with the highest `mi_shuffle_z` are the strongest spatial/topographic markers.
2. Probes with low normalized spatial entropy are spatially restricted.
3. Probes with high mean JSD are topographically unique compared with other probes.
4. Density-stratified MI helps rule out uneven RGC density or sampling as the only explanation.
5. Probe clusters represent reusable spatial motifs for control/injury comparisons.
